# Rékolte — Notebook 1: Satellite Feature Extraction (NDVI + Rainfall + Temperature)
**Project:** Predicting Sugarcane Yield in Mauritius Using Satellite Imagery  
**Student:** Yashvin Booputh (M01006629)  
**Module:** CST3990 — Spring 2025/2026

---

## What this notebook does
Extracts three groups of satellite and climate features for all 5 Mauritius sugarcane regions
across 18 crushing seasons (2008–2025) using Google Earth Engine:

| Feature group | Source | Resolution | Columns |
|--------------|--------|------------|--------|
| Monthly NDVI (Jun–Dec) + aggregates | Landsat 7/8 + Sentinel-2 | 10–30 m | 10 cols |
| Monthly CHIRPS rainfall (Jun–Dec) | CHIRPS Daily | ~5 km | 7 cols |
| Monthly ERA5-Land temperature (Jun–Dec) | ERA5-Land Monthly | ~11 km | 7 cols |

**Total features extracted:** 24 per region per season  
(Notebook 2 adds region, satellite, and surface_prev one-hot/derived features to reach 31 total)

### Satellite strategy
| Years | Satellite | GEE Collection |
|-------|-----------|----------------|
| 2008–2012 | Landsat 7 ETM+ | `USGS/LE07/C02/T1_L2` |
| 2013–2016 | Landsat 8 OLI | `USGS/LC08/C02/T1_L2` |
| 2017–2025 | Sentinel-2 SR Harmonised | `COPERNICUS/S2_SR_HARMONIZED` |

> **Why Landsat 8 for 2013–2016?**  
> Sentinel-2 SR in GEE only starts in 2017. Using Landsat 8 avoids mixing Surface Reflectance
> and Top-of-Atmosphere processing levels, keeping NDVI values consistent within each sensor era.

**Season window:** June 1 – December 31 (Mauritius sugarcane crushing season)  
**GEE project:** `rekolte-491422`  
**Output:** `satellite_features.csv` → `My Drive/CST3990 - Final Year Project/model/`

### NDVI features extracted per region per season
| Column | Description |
|--------|-------------|
| `ndvi_june` … `ndvi_dec` | Monthly mean NDVI (7 values) |
| `ndvi_mean` | Season mean NDVI |
| `ndvi_max` | Peak NDVI during season |
| `ndvi_cumulative` | Sum of NDVI observations (proxy for integrated photosynthesis) |
| `observation_count` | Number of valid (cloud-free) images used |
| `satellite` | Sensor used: `landsat7`, `landsat8`, or `sentinel2` |

### Climate features extracted per region per season
| Column | Source | Description |
|--------|--------|-------------|
| `rainfall_june` … `rainfall_dec` | CHIRPS Daily | Monthly total rainfall (mm) |
| `temp_june` … `temp_dec` | ERA5-Land Monthly | Mean 2 m air temperature (°C) |


## Cell 1 — Install packages
Run this once. Restart runtime if prompted.

In [ ]:
!pip install earthengine-api --quiet
print('Packages ready.')

## Cell 2 — Imports

In [ ]:
import ee
import pandas as pd
import numpy as np
import os
import json
import time
from datetime import datetime

print('Imports OK.')

## Cell 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify the paths we need exist
DRIVE_BASE = '/content/drive/MyDrive/CST3990 - Final Year Project'
OUTPUT_DIR = f'{DRIVE_BASE}/model'
OUTPUT_CSV = f'{OUTPUT_DIR}/satellite_features.csv'
CHECKPOINT_CSV = f'{OUTPUT_DIR}/satellite_features_checkpoint.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR}')

## Cell 4 — Authenticate & Initialise Google Earth Engine
This will open a browser link. Follow it, grant permission, and paste the code back here.

In [ ]:
ee.Authenticate()
ee.Initialize(project='rekolte-491422')
print('GEE initialised successfully.')

## Cell 5 — Load Region Assets from GEE Cloud
Each region's boundary is a MultiPolygon of individually mapped sugarcane fields,
uploaded as cloud assets under project `rekolte-491422`.

In [ ]:
ASSET_BASE = 'projects/rekolte-491422/assets'
REGION_NAMES = ['NORD', 'SUD', 'EST', 'OUEST', 'CENTRE']

# Load each region as a FeatureCollection and dissolve to a single geometry
region_geometries = {}
for name in REGION_NAMES:
    fc = ee.FeatureCollection(f'{ASSET_BASE}/{name}_boundary')
    region_geometries[name] = fc.geometry()
    print(f'Loaded {name}_boundary')

print('\nAll 5 region assets loaded.')

## Cell 6 — Satellite Preprocessing Functions

### Cloud masking
- **Landsat 7/8 Collection 2 Level 2:** QA_PIXEL band — mask cloud (bit 5), cloud shadow (bit 3), snow/ice (bit 4)
- **Sentinel-2 SR Harmonized:** QA60 band — mask opaque clouds (bit 10) and cirrus (bit 11)

### Scale factors (Landsat C2L2 only)
Landsat Collection 2 Level 2 SR values must be scaled: `reflectance = pixel_value × 0.0000275 − 0.2`
This offset matters for NDVI accuracy — Sentinel-2 NDVI ratio is scale-invariant.

In [ ]:
# ── Landsat shared ────────────────────────────────────────────────────────────

def apply_scale_factors_landsat(image):
    """Apply Collection 2 Level 2 scale factors to SR bands."""
    optical = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    return image.addBands(optical, None, True)

def mask_clouds_landsat(image):
    """Mask clouds, cloud shadows, and snow using QA_PIXEL."""
    qa = image.select('QA_PIXEL')
    cloud        = qa.bitwiseAnd(1 << 5).eq(0)   # cloud
    cloud_shadow = qa.bitwiseAnd(1 << 3).eq(0)   # cloud shadow
    snow         = qa.bitwiseAnd(1 << 4).eq(0)   # snow/ice
    mask = cloud.And(cloud_shadow).And(snow)
    return image.updateMask(mask)

# ── Landsat 7 ─────────────────────────────────────────────────────────────────

def preprocess_landsat7(image):
    """Scale → cloud mask → NDVI for Landsat 7 ETM+ (NIR=SR_B4, Red=SR_B3)."""
    img = mask_clouds_landsat(apply_scale_factors_landsat(image))
    ndvi = img.normalizedDifference(['SR_B4', 'SR_B3']).rename('NDVI')
    return img.addBands(ndvi)

# ── Landsat 8 ─────────────────────────────────────────────────────────────────

def preprocess_landsat8(image):
    """Scale → cloud mask → NDVI for Landsat 8 OLI (NIR=SR_B5, Red=SR_B4)."""
    img = mask_clouds_landsat(apply_scale_factors_landsat(image))
    ndvi = img.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    return img.addBands(ndvi)

# ── Sentinel-2 ────────────────────────────────────────────────────────────────

def mask_clouds_sentinel2(image):
    """Mask opaque clouds and cirrus using QA60 band."""
    qa = image.select('QA60')
    opaque_clouds = qa.bitwiseAnd(1 << 10).eq(0)
    cirrus        = qa.bitwiseAnd(1 << 11).eq(0)
    return image.updateMask(opaque_clouds.And(cirrus))

def preprocess_sentinel2(image):
    """Cloud mask → NDVI for Sentinel-2 SR (NIR=B8, Red=B4).
    Note: normalizedDifference cancels the 10000 scale factor."""
    img = mask_clouds_sentinel2(image)
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return img.addBands(ndvi)

# ── Lookup tables ─────────────────────────────────────────────────────────────

# FIX: correct prefix is LANDSAT/ not USGS/
COLLECTIONS = {
    'landsat7':  'LANDSAT/LE07/C02/T1_L2',
    'landsat8':  'LANDSAT/LC08/C02/T1_L2',
    'sentinel2': 'COPERNICUS/S2_SR_HARMONIZED'
}

PREPROCESSORS = {
    'landsat7':  preprocess_landsat7,
    'landsat8':  preprocess_landsat8,
    'sentinel2': preprocess_sentinel2
}

# Landsat = 30m, Sentinel-2 = 10m (using B8 NIR band)
SCALES = {
    'landsat7':  30,
    'landsat8':  30,
    'sentinel2': 10
}

print('Preprocessing functions defined.')

## Cell 7 — Season-to-Satellite Mapping

In [ ]:
SEASON_SATELLITE = {}
for y in range(2008, 2013):  # 2008-2012: Landsat 7
    SEASON_SATELLITE[y] = 'landsat7'
for y in range(2013, 2017):  # 2013-2016: Landsat 8 (cleaner SR than S2 TOA)
    SEASON_SATELLITE[y] = 'landsat8'
for y in range(2017, 2026):  # 2017-2025: Sentinel-2 SR Harmonized
    SEASON_SATELLITE[y] = 'sentinel2'

print('Season → Satellite mapping:')
for season, sat in SEASON_SATELLITE.items():
    print(f'  {season}: {sat}')

## Cell 8 — NDVI Extraction Function

**Strategy:** For each (season, region), retrieve all valid cloud-masked images,
compute the spatial mean NDVI over the region per image, then aggregate:
- Monthly means (June–December)
- Season mean, max, cumulative

We do this in **2 GEE API calls per region-season** (dates list + NDVI values list)
and process everything in Python — minimising round-trips.

In [ ]:
MONTH_KEYS = {6: 'june', 7: 'july', 8: 'aug', 9: 'sep', 10: 'oct', 11: 'nov', 12: 'dec'}

def get_ndvi_collection(season, satellite, region_geom):
    """Return cloud-masked NDVI ImageCollection for a season window."""
    return (
        ee.ImageCollection(COLLECTIONS[satellite])
        .filterDate(f'{season}-06-01', f'{season}-12-31')
        .filterBounds(region_geom)
        .map(PREPROCESSORS[satellite])
        .select('NDVI')
    )

def empty_record(season, region, satellite):
    """Return a record with all NDVI values as None (no data)."""
    rec = {'season': season, 'region': region, 'satellite': satellite, 'observation_count': 0}
    for key in MONTH_KEYS.values():
        rec[f'ndvi_{key}'] = None
    rec['ndvi_mean']       = None
    rec['ndvi_max']        = None
    rec['ndvi_cumulative'] = None
    return rec

def extract_features(season, region_name, region_geom):
    """
    Extract all NDVI features for one (season, region) pair.
    Returns a dict with monthly NDVI + season aggregates.
    """
    satellite = SEASON_SATELLITE[season]
    scale     = SCALES[satellite]
    collection = get_ndvi_collection(season, satellite, region_geom)

    # Check image count server-side before pulling data
    size = collection.size().getInfo()
    if size == 0:
        return empty_record(season, region_name, satellite)

    # Map each image to a Feature with date + NDVI mean over region
    def img_to_feature(img):
        mean_val = img.reduceRegion(
            reducer   = ee.Reducer.mean(),
            geometry  = region_geom,
            scale     = scale,
            maxPixels = 1e10,
            bestEffort= True
        ).get('NDVI')
        return ee.Feature(None, {
            'img_date': img.date().millis(),
            'ndvi_val': mean_val
        })

    fc_stats  = ee.FeatureCollection(collection.map(img_to_feature))
    data_list = fc_stats.getInfo()['features']

    # FIX: use .get() with None default — when all pixels are cloud-masked,
    # GEE omits the null property key entirely from the serialised Feature.
    dates_ms  = [f['properties'].get('img_date', None) for f in data_list]
    ndvi_vals = [f['properties'].get('ndvi_val',  None) for f in data_list]

    # Build pandas DataFrame for aggregation
    df = pd.DataFrame({'date': pd.to_datetime(dates_ms, unit='ms'), 'ndvi': ndvi_vals})
    df = df.dropna(subset=['ndvi'])
    df['month'] = df['date'].dt.month

    rec = {'season': season, 'region': region_name, 'satellite': satellite,
           'observation_count': len(df)}

    # Monthly means
    for month_num, month_key in MONTH_KEYS.items():
        monthly = df.loc[df['month'] == month_num, 'ndvi']
        rec[f'ndvi_{month_key}'] = float(monthly.mean()) if len(monthly) > 0 else None

    # Season aggregates
    if len(df) > 0:
        rec['ndvi_mean']       = float(df['ndvi'].mean())
        rec['ndvi_max']        = float(df['ndvi'].max())
        rec['ndvi_cumulative'] = float(df['ndvi'].sum())
    else:
        rec['ndvi_mean']       = None
        rec['ndvi_max']        = None
        rec['ndvi_cumulative'] = None

    return rec

print('Extraction function defined.')

## Cell 9 — Run Extraction (18 seasons × 5 regions = 90 combinations)

**Expected runtime:** ~20–40 minutes (GEE compute + network latency).  
Progress is saved every 5 records to a checkpoint file so you can resume if interrupted.

In [ ]:
SEASONS = list(range(2008, 2026))
total   = len(SEASONS) * len(REGION_NAMES)
results = []
done    = 0

print(f'Starting extraction: {total} combinations\n')
start_time = time.time()

for season in SEASONS:
    for region in REGION_NAMES:
        done += 1
        elapsed = time.time() - start_time
        eta = (elapsed / done) * (total - done) if done > 1 else 0
        print(f'[{done:2d}/{total}] {season} {region:<7} ', end='', flush=True)

        try:
            rec = extract_features(season, region, region_geometries[region])
            results.append(rec)
            print(f'obs={rec["observation_count"]:3d}  '
                  f'ndvi_mean={rec["ndvi_mean"]:.3f}' if rec['ndvi_mean'] else
                  f'obs=0  ndvi_mean=None')
        except Exception as e:
            print(f'ERROR: {e}')
            results.append(empty_record(season, region, SEASON_SATELLITE[season]))

        # Checkpoint every 5 records
        if done % 5 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False)
            print(f'  → checkpoint saved ({done}/{total}, ETA {eta/60:.1f} min)')

# Final save
df_sat = pd.DataFrame(results)
df_sat.to_csv(OUTPUT_CSV, index=False)
elapsed_total = (time.time() - start_time) / 60
print(f'\n✓ Done in {elapsed_total:.1f} min. Saved {len(df_sat)} records to:')
print(f'  {OUTPUT_CSV}')

## Cell 10 — CHIRPS Rainfall + ERA5 Temperature Extraction

Extracts two new feature groups for each region × season:

| Feature | Source | Resolution | Columns |
|---------|--------|-----------|----------|
| Monthly rainfall | CHIRPS Daily (`UCSB-CHG/CHIRPS/DAILY`) | ~5 km | `rainfall_june` … `rainfall_dec` |
| Monthly temperature | ERA5-Land Monthly (`ECMWF/ERA5_LAND/MONTHLY_AGGR`) | ~11 km | `temp_june` … `temp_dec` |

**Why CHIRPS for rainfall?** 5 km resolution captures the North/West (dry, leeward) vs East/South/Centre (wet, windward) gradient that drives yield differences between regions.  
**Why ERA5 for temperature?** Temperature is spatially smooth — ERA5 is accurate at 11 km. CHIRPS has no temperature band.

**Expected runtime:** ~25–40 minutes (90 GEE calls × 14 months each).

In [ ]:
import calendar

SEASON_MONTHS_LIST = [6, 7, 8, 9, 10, 11, 12]
MONTH_KEY_LIST     = ['june', 'july', 'aug', 'sep', 'oct', 'nov', 'dec']

# Output paths
CLIMATE_CHECKPOINT = f'{OUTPUT_DIR}/climate_checkpoint.csv'
CLIMATE_CSV        = f'{OUTPUT_DIR}/climate_features.csv'


def extract_chirps_monthly(year, region_geom):
    """
    CHIRPS daily precipitation -> monthly totals (mm) for June-December.
    CHIRPS unit: mm/day  ->  sum over days in month  =  mm/month.
    """
    monthly = {}
    for month, key in zip(SEASON_MONTHS_LIST, MONTH_KEY_LIST):
        _, last_day = calendar.monthrange(year, month)
        col_name = f'rainfall_{key}'
        try:
            col = (
                ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
                .filterDate(
                    f'{year}-{month:02d}-01',
                    f'{year}-{month:02d}-{last_day:02d}'
                )
                .select('precipitation')
            )
            if col.size().getInfo() == 0:
                monthly[col_name] = None
                continue
            total = col.sum()
            result = total.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region_geom,
                scale=5566,          # CHIRPS native: 0.05 deg ~ 5566 m at equator
                maxPixels=1e9,
                bestEffort=True
            ).getInfo()
            val = result.get('precipitation')
            monthly[col_name] = round(val, 2) if val is not None else None
        except Exception as e:
            print(f'      CHIRPS {year}-{month:02d}: {e}')
            monthly[col_name] = None
    return monthly


def extract_era5_temp_monthly(year, region_geom):
    """
    ERA5-Land monthly mean 2 m temperature (degrees C) for June-December.
    ERA5 unit: Kelvin  ->  subtract 273.15 for Celsius.
    """
    monthly = {}
    for month, key in zip(SEASON_MONTHS_LIST, MONTH_KEY_LIST):
        next_month = month + 1 if month < 12 else 1
        next_year  = year if month < 12 else year + 1
        col_name   = f'temp_{key}'
        try:
            col = (
                ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
                .filterDate(
                    f'{year}-{month:02d}-01',
                    f'{next_year}-{next_month:02d}-01'
                )
                .select('temperature_2m')
            )
            if col.size().getInfo() == 0:
                monthly[col_name] = None
                continue
            img = col.mean()
            result = img.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=region_geom,
                scale=11132,         # ERA5-Land native: 0.1 deg ~ 11132 m at equator
                maxPixels=1e9,
                bestEffort=True
            ).getInfo()
            temp_k = result.get('temperature_2m')
            monthly[col_name] = round(temp_k - 273.15, 3) if temp_k is not None else None
        except Exception as e:
            print(f'      ERA5 temp {year}-{month:02d}: {e}')
            monthly[col_name] = None
    return monthly


print('Climate extraction functions ready.')

## Cell 11 — Run Climate Extraction (90 combinations)

Saves a checkpoint every 5 records. If interrupted, re-run from this cell —
already-extracted rows will be in the checkpoint file.

In [ ]:
climate_results = []
total_c = len(SEASONS) * len(REGION_NAMES)
done_c  = 0
start_c = time.time()

print(f'Climate extraction: {total_c} combinations\n')

for season in SEASONS:
    for region in REGION_NAMES:
        done_c += 1
        elapsed = time.time() - start_c
        eta = (elapsed / done_c) * (total_c - done_c) if done_c > 1 else 0
        print(f'[{done_c:2d}/{total_c}] {season} {region:<7} ', end='', flush=True)

        try:
            chirps = extract_chirps_monthly(season, region_geometries[region])
            era5   = extract_era5_temp_monthly(season, region_geometries[region])

            rec = {'season': season, 'region': region}
            rec.update(chirps)
            rec.update(era5)

            # Season aggregates (sanity check only, not extra model features)
            rain_vals = [v for v in chirps.values() if v is not None]
            temp_vals = [v for v in era5.values()   if v is not None]
            rec['rainfall_total'] = round(sum(rain_vals), 1) if rain_vals else None
            rec['temp_mean']      = round(sum(temp_vals) / len(temp_vals), 3) if temp_vals else None

            climate_results.append(rec)
            rain_str = f"{rec['rainfall_total']:.0f}mm" if rec['rainfall_total'] else 'None'
            temp_str = f"{rec['temp_mean']:.1f}C"       if rec['temp_mean']      else 'None'
            print(f'rain={rain_str}  temp={temp_str}  ETA={eta/60:.1f}min')

        except Exception as e:
            print(f'ERROR: {e}')
            rec = {'season': season, 'region': region}
            for key in MONTH_KEY_LIST:
                rec[f'rainfall_{key}'] = None
                rec[f'temp_{key}']     = None
            rec['rainfall_total'] = None
            rec['temp_mean']      = None
            climate_results.append(rec)

        if done_c % 5 == 0:
            pd.DataFrame(climate_results).to_csv(CLIMATE_CHECKPOINT, index=False)
            print(f'  [checkpoint saved — {done_c}/{total_c}]')

df_climate = pd.DataFrame(climate_results)
df_climate.to_csv(CLIMATE_CSV, index=False)
print(f'\nSaved {len(df_climate)} rows -> {CLIMATE_CSV}')
print(f'Missing rainfall: {df_climate[[f"rainfall_{k}" for k in MONTH_KEY_LIST]].isnull().sum().sum()} values')
print(f'Missing temp:     {df_climate[[f"temp_{k}" for k in MONTH_KEY_LIST]].isnull().sum().sum()} values')

## Cell 12 — Merge NDVI + Climate → satellite_features.csv

Combines NDVI features (Cell 9) with CHIRPS rainfall and ERA5 temperature into one file.  
This is the final output that Notebook 2 reads.

In [ ]:
# Load NDVI features saved by Cell 9
df_ndvi = pd.read_csv(OUTPUT_CSV)

# Merge on season + region (left join keeps all 90 NDVI rows)
df_combined = pd.merge(df_ndvi, df_climate, on=['season', 'region'], how='left')

print(f'NDVI features:    {df_ndvi.shape[0]} rows x {df_ndvi.shape[1]} cols')
print(f'Climate features: {df_climate.shape[0]} rows x {df_climate.shape[1]} cols')
print(f'Combined:         {df_combined.shape[0]} rows x {df_combined.shape[1]} cols')

missing = df_combined.isnull().sum()
missing = missing[missing > 0]
if len(missing):
    print(f'\nMissing values:\n{missing}')
else:
    print('\nNo missing values — all 90 rows complete.')

# Overwrite satellite_features.csv with the enriched version
df_combined.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved -> {OUTPUT_CSV}')
print(f'Final columns ({len(df_combined.columns)}):')
for col in df_combined.columns:
    print(f'  {col}')

## Cell 13 — Data Quality Check

In [ ]:
import matplotlib.pyplot as plt

df_sat = pd.read_csv(OUTPUT_CSV)
print('Shape:', df_sat.shape)
print('\nMissing values per column:')
print(df_sat.isnull().sum())

print('\nObservations per satellite:')
print(df_sat.groupby('satellite')['observation_count'].describe())

# Heatmap: observation count by season × region
pivot = df_sat.pivot(index='season', columns='region', values='observation_count')
fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGn')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
plt.colorbar(im, ax=ax, label='Valid observations')
ax.set_title('Valid cloud-free observations per season × region')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, str(pivot.iloc[i, j]), ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/observation_heatmap.png', dpi=150)
plt.show()
print('Heatmap saved.')

## Cell 14 — NDVI Trends by Region
Sanity check: NDVI should follow seasonal patterns and vary meaningfully across regions.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)

for ax, region in zip(axes, REGION_NAMES):
    subset = df_sat[df_sat['region'] == region].sort_values('season')
    ax.plot(subset['season'], subset['ndvi_mean'], marker='o', label='mean')
    ax.fill_between(subset['season'],
                    subset['ndvi_mean'] - 0.02,
                    subset['ndvi_mean'] + 0.02,
                    alpha=0.2)
    ax.set_title(region)
    ax.set_xlabel('Season')
    ax.tick_params(axis='x', rotation=45)

axes[0].set_ylabel('NDVI (season mean)')
fig.suptitle('Season-mean NDVI by Region (2008–2025)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ndvi_trends.png', dpi=150)
plt.show()

## Cell 15 — Done

`satellite_features.csv` is now saved to `MODEL_DIR` on your Google Drive.

**Verify the output:**
- Shape should be **90 rows × 26+ columns** (18 seasons × 5 regions)
- Columns: `season`, `region`, `satellite`, `observation_count`,
  `ndvi_june`…`ndvi_dec`, `ndvi_mean`, `ndvi_max`, `ndvi_cumulative`,
  `rainfall_june`…`rainfall_dec`, `temp_june`…`temp_dec`
- No region/season should be entirely missing

**Next step:** Open `02_model_training_final.ipynb` and point `DATA_DIR` / `MODEL_DIR`
to the same Google Drive folder. Notebook 2 reads `satellite_features.csv` directly
from Drive and produces the trained model files.


In [ ]:
df_final = pd.read_csv(OUTPUT_CSV)
print(f'satellite_features.csv: {df_final.shape[0]} rows × {df_final.shape[1]} columns')
print(f'Seasons:  {sorted(df_final["season"].unique())}')
print(f'Regions:  {sorted(df_final["region"].unique())}')
print(f'Missing:  {df_final.isnull().sum().sum()} total NaN values')
print('\nFirst 5 rows:')
df_final.head()